PMI-based similarity

In [1]:
#setup

from pathlib import Path
from collections import Counter, defaultdict
import math
import random
import re

import pandas as pd

pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("../data/Sentences_50Agree.txt")
OUTPUT_DIR = Path("../results/task4_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

In [2]:
#load dataset

rows = []

with open(DATA_PATH, "r", encoding="latin-1") as f:
    for line in f:
        line = line.strip()
        if line:
            text, label = line.rsplit("@", 1)
            rows.append({
                "text": text.strip(),
                "label": label.strip()
            })

df = pd.DataFrame(rows)

print("Dataset loaded successfully.")
print("Number of examples:", len(df))
print("Labels:", sorted(df["label"].unique()))

df.head()

Dataset loaded successfully.
Number of examples: 4846
Labels: ['negative', 'neutral', 'positive']


,text,label
0,"According to Gran , the company has no plans to move all production to Russia , although that is where the company i...",neutral
1,"Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies work...",neutral
2,The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; con...,negative
3,With the new production plant the company would increase its capacity to meet the expected increase in demand and wo...,positive
4,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth ...",positive


In [3]:
from nltk.tokenize import RegexpTokenizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

tokenizer = RegexpTokenizer(r"[A-Za-z]+")

stop_words = set(ENGLISH_STOP_WORDS)

#same words i kept in preprocessing
words_to_keep = {
    "no", "not", "nor",
    "up", "down",
}

stop_words = stop_words - words_to_keep

def tokenize_pmi(text):
    tokens = tokenizer.tokenize(text.lower())
    
    tokens = [
        token for token in tokens
        if token not in stop_words
        and len(token) > 1
    ]
    
    return tokens

df["tokens"] = df["text"].apply(tokenize_pmi)

df[["text", "tokens"]].head()

,text,tokens
0,"According to Gran , the company has no plans to move all production to Russia , although that is where the company i...","[according, gran, company, no, plans, production, russia, company, growing]"
1,"Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies work...","[technopolis, plans, develop, stages, area, no, square, meters, order, host, companies, working, computer, technolog..."
2,The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; con...,"[international, electronic, industry, company, elcoteq, laid, tens, employees, tallinn, facility, contrary, earlier,..."
3,With the new production plant the company would increase its capacity to meet the expected increase in demand and wo...,"[new, production, plant, company, increase, capacity, meet, expected, increase, demand, improve, use, raw, materials..."
4,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth ...","[according, company, updated, strategy, years, basware, targets, long, term, net, sales, growth, range, operating, p..."


In [27]:
# we keep only words that appear at least 5 times

MIN_WORD_FREQ = 5

all_tokens = [
    token
    for tokens in df["tokens"]
    for token in tokens
]

word_counts = Counter(all_tokens)

vocabulary = {
    word for word, count in word_counts.items()
    if count >= MIN_WORD_FREQ
}

print("Vocabulary size before filtering:", len(word_counts))
print("Vocabulary size after filtering:", len(vocabulary))
print("Removed vocabulary items:", len(word_counts) - len(vocabulary))

Vocabulary size before filtering: 8805
Vocabulary size after filtering: 1976
Removed vocabulary items: 6829


In [29]:
#co occurrence counts

WINDOW_SIZE = 1

cooccurrence_counts = Counter()

for tokens in df["tokens"]:
    for i, target_word in enumerate(tokens):
        
        if target_word not in vocabulary:
            continue
        
        left = max(0, i - WINDOW_SIZE)
        right = min(len(tokens), i + WINDOW_SIZE + 1)
        
        for j in range(left, right):
            if i == j:
                continue
            
            context_word = tokens[j]
            
            if context_word not in vocabulary:
                continue
            
            cooccurrence_counts[(target_word, context_word)] += 1

target_counts = Counter()
context_counts = Counter()

for (target_word, context_word), count in cooccurrence_counts.items():
    target_counts[target_word] += count
    context_counts[context_word] += count

total_cooccurrences = sum(cooccurrence_counts.values())

print("Number of co-occurrence pairs:", len(cooccurrence_counts))
print("Total co-occurrences:", total_cooccurrences)

Number of co-occurrence pairs: 40443
Total co-occurrences: 70608


PMI(w,c) = log2( P(w,c) / (P(w) P(c)) )

PMI(w,c) = log2( count(w,c) * total / (count(w) * count(c)) )

In [30]:
#pmi vectors

pmi_vectors = defaultdict(dict)

for (target_word, context_word), cooc_count in cooccurrence_counts.items():
    numerator = cooc_count * total_cooccurrences
    denominator = target_counts[target_word] * context_counts[context_word]
    
    pmi = math.log2(numerator / denominator)
    
    pmi_vectors[target_word][context_word] = pmi

print("Number of PMI vectors:", len(pmi_vectors))

Number of PMI vectors: 1976


In [32]:
#vector cosine similarity
def cosine_similarity(vector_a, vector_b):
    shared_contexts = set(vector_a.keys()) & set(vector_b.keys())
    
    if not shared_contexts:
        return 0.0
    
    dot_product = sum(
        vector_a[context] * vector_b[context]
        for context in shared_contexts
    )
    
    norm_a = math.sqrt(sum(value ** 2 for value in vector_a.values()))
    norm_b = math.sqrt(sum(value ** 2 for value in vector_b.values()))
    
    if norm_a == 0 or norm_b == 0:
        return 0.0
    
    return dot_product / (norm_a * norm_b)


#finds 10 most similar words
def most_similar_words(target_word, top_k=5):
    target_vector = pmi_vectors[target_word]
    
    similarities = []
    
    for other_word, other_vector in pmi_vectors.items():
        if other_word == target_word:
            continue
        
        similarity = cosine_similarity(target_vector, other_vector)
        similarities.append((other_word, similarity))
    
    similarities = sorted(similarities, key=lambda x: x[1], reverse=True)
    
    return similarities[:top_k]


In [33]:
#random words 

MIN_CONTEXTS = 5
N_RANDOM_WORDS = 10

candidate_words = [
    word
    for word, vector in pmi_vectors.items()
    if len(vector) >= MIN_CONTEXTS
]

random_words = random.sample(candidate_words, N_RANDOM_WORDS)

print("Number of candidate words:", len(candidate_words))
random_words

Number of candidate words: 1879


['ministry',
 'working',
 'programs',
 'needs',
 'countries',
 'changed',
 'introduce',
 'fifth',
 'forest',
 'dec']

In [34]:
#most similar words for each random word

TOP_K = 5

similarity_results = []

for target_word in random_words:
    similar_words = most_similar_words(target_word, top_k=TOP_K)
    
    similarity_results.append({
        "target_word": target_word,
        "most_similar_words": ", ".join(word for word, score in similar_words),
        "similarity_scores": ", ".join(f"{score:.3f}" for word, score in similar_words)
    })

similarity_results_df = pd.DataFrame(similarity_results)

similarity_results_df

,target_word,most_similar_words,similarity_scores
0,ministry,"infrastructure, sectors, republic, great, sawmill","0.290, 0.241, 0.240, 0.221, 0.202"
1,working,"produced, extended, independent, upm, restructuring","0.217, 0.183, 0.168, 0.144, 0.137"
2,programs,"black, ssh, operational, independent, model","0.177, 0.175, 0.162, 0.156, 0.156"
3,needs,"drive, gives, overview, comparable, holds","0.182, 0.176, 0.174, 0.150, 0.149"
4,countries,"regions, offices, players, northern, north","0.227, 0.200, 0.198, 0.176, 0.160"
5,changed,"magazines, places, fresh, closed, enso","0.244, 0.188, 0.181, 0.177, 0.158"
6,introduce,"version, step, clean, export, implementation","0.384, 0.342, 0.294, 0.287, 0.258"
7,fifth,"letter, took, consumption, transaction, lost","0.228, 0.203, 0.195, 0.193, 0.186"
8,forest,"process, paper, profile, heavy, construction","0.232, 0.198, 0.163, 0.155, 0.153"
9,dec,"nov, feb, oct, jan, alexandria","0.450, 0.429, 0.402, 0.369, 0.310"


In [35]:
similarity_results_path = OUTPUT_DIR / "pmi_similarity_results.csv"

similarity_results_df.to_csv(similarity_results_path, index=False)

print("Saved summary results to:", similarity_results_path)

Saved summary results to: ..\results\task4_outputs\pmi_similarity_results.csv
